<a href="https://colab.research.google.com/github/prajwalmotagi/imdb-movie-recommender/blob/main/imdb_movie_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDb Movie Recommender
Built a weighted rating system to fix vote-count bias in IMDb ratings.
Loads 5,000 real movies, calculates fair scores using Bayesian-style
weighting, and ranks movies by trustworthiness of their rating.

In [13]:
movies = [
    {"name": "Tiny Indie Film", "rating": 9.5, "votes": 200},
    {"name": "The Shawshank Redemption", "rating": 9.3, "votes": 2800000},
    {"name": "Random Short Film", "rating": 9.6, "votes": 45},
    {"name": "The Dark Knight", "rating": 9.0, "votes": 2700000},
    {"name": "The Outsider", "rating": 6.3, "votes": 31746},
    {"name": "The Kashmir Files", "rating": 8.5, "votes": 579183},
    {"name": "Parasite", "rating": 8.5, "votes": 1171802}
]

print(movies)

[{'name': 'Tiny Indie Film', 'rating': 9.5, 'votes': 200}, {'name': 'The Shawshank Redemption', 'rating': 9.3, 'votes': 2800000}, {'name': 'Random Short Film', 'rating': 9.6, 'votes': 45}, {'name': 'The Dark Knight', 'rating': 9.0, 'votes': 2700000}, {'name': 'The Outsider', 'rating': 6.3, 'votes': 31746}, {'name': 'The Kashmir Files', 'rating': 8.5, 'votes': 579183}, {'name': 'Parasite', 'rating': 8.5, 'votes': 1171802}]


In [14]:
C = 7.0      # the "default expected rating" every movie starts near
m = 25000    # how many votes it takes before we mostly trust its own rating

def weighted_rating(votes, rating):
    return (votes / (votes + m)) * rating + (m / (votes + m)) * C

print(weighted_rating(200, 9.5))
print(weighted_rating(2800000, 9.3))

7.01984126984127
9.279646017699115


In [15]:
for movie in movies:
    movie["weighted_score"] = weighted_rating(movie["votes"], movie["rating"])

for movie in movies:
    print(movie["name"], "→", round(movie["weighted_score"], 2))

Tiny Indie Film → 7.02
The Shawshank Redemption → 9.28
Random Short Film → 7.0
The Dark Knight → 8.98
The Outsider → 6.61
The Kashmir Files → 8.44
Parasite → 8.47


In [16]:
ranked = sorted(movies, key=lambda m: m["weighted_score"], reverse=True)

print("--- FINAL RANKING ---")
for i, movie in enumerate(ranked):
    print(f"{i+1}. {movie['name']} | rating: {movie['rating']} | weighted: {round(movie['weighted_score'], 2)}")

--- FINAL RANKING ---
1. The Shawshank Redemption | rating: 9.3 | weighted: 9.28
2. The Dark Knight | rating: 9.0 | weighted: 8.98
3. Parasite | rating: 8.5 | weighted: 8.47
4. The Kashmir Files | rating: 8.5 | weighted: 8.44
5. Tiny Indie Film | rating: 9.5 | weighted: 7.02
6. Random Short Film | rating: 9.6 | weighted: 7.0
7. The Outsider | rating: 6.3 | weighted: 6.61


In [18]:
def recommend(movie_list, top_n=3):
    for movie in movie_list:
        movie["weighted_score"] = weighted_rating(
            movie["votes"], movie["rating"]
        )
    ranked = sorted(
        movie_list,
        key=lambda m: m["weighted_score"],
        reverse=True
    )
    print(f"--- TOP {top_n} RECOMMENDATIONS ---")
    for i, movie in enumerate(ranked[:top_n]):
        print(f"{i+1}. {movie['name']} | weighted: {round(movie['weighted_score'], 2)}")

recommend(movies, top_n=3)

--- TOP 3 RECOMMENDATIONS ---
1. The Shawshank Redemption | weighted: 9.28
2. The Dark Knight | weighted: 8.98
3. Parasite | weighted: 8.47


In [20]:
import pandas as pd

df = pd.read_csv("results_with_crew.csv")

print(df.shape)
print(df.head())

(5000, 12)
      tconst                                   primaryTitle  startYear  rank  \
0  tt0111161                       The Shawshank Redemption       1994     1   
1  tt0068646                                  The Godfather       1972     2   
2  tt0468569                                The Dark Knight       2008     3   
3  tt0167260  The Lord of the Rings: The Return of the King       2003     4   
4  tt0108052                               Schindler's List       1993     5   

   averageRating  numVotes  runtimeMinutes             directors  \
0            9.3   3198304             142        Frank Darabont   
1            9.2   2231582             175  Francis Ford Coppola   
2            9.1   3179134             152     Christopher Nolan   
3            9.0   2169878             201         Peter Jackson   
4            9.0   1588096             195      Steven Spielberg   

                                             writers  \
0                       Frank Darabont, Ste

In [22]:
C = df["averageRating"].mean()
m = df["numVotes"].quantile(0.25)

print("Average rating across all movies:", round(C, 2))
print("Minimum votes threshold:", round(m, 2))

Average rating across all movies: 7.18
Minimum votes threshold: 40005.0


In [23]:
def weighted_rating_df(row):
    v = row["numVotes"]
    R = row["averageRating"]
    return (v / (v + m)) * R + (m / (v + m)) * C

df["weighted_score"] = df.apply(weighted_rating_df, axis=1)

top_movies = df.sort_values("weighted_score", ascending=False).head(10)

print(top_movies[["primaryTitle", "averageRating", "numVotes", "weighted_score"]].to_string(index=False))

                                     primaryTitle  averageRating  numVotes  weighted_score
                         The Shawshank Redemption            9.3   3198304        9.273768
                                    The Godfather            9.2   2231582        9.164366
                                  The Dark Knight            9.1   3179134        9.076098
    The Lord of the Rings: The Return of the King            9.0   2169878        8.966992
                                 Schindler's List            9.0   1588096        8.955197
                            The Godfather Part II            9.0   1497791        8.952566
                                     12 Angry Men            9.0    987283        8.928993
The Lord of the Rings: The Fellowship of the Ring            8.9   2212021        8.869386
                                        Inception            8.8   2826603        8.777345
                                       Fight Club            8.8   2618882        8.775575